# Multi Player Mode

I know, sometimes we get overly popular and it could happen that we're evaluating more than one person. 

In the multi player mode, you can compare the scores for two or more people. The higher the score the better the relationship.

## Build Functions

In [45]:
def calculate_individual_score(name, positive_traits, negative_traits, baseline):
    # numerator: sum of Ptrait * Wp
    numerator = sum(score * weight for (score, weight) in positive_traits.values())
    # denominator: sum of Ntrait * Wn
    denominator = sum(score * weight for (score, weight) in negative_traits.values())
    
    # Validation checks
    if denominator == 0:
        return None, f"Error for {name}: Denominator is zero. Cannot compute score."
    
    total_positive_weight = sum(weight for (_, weight) in positive_traits.values())
    total_negative_weight = sum(weight for (_, weight) in negative_traits.values())
    
    eps = 1e-6
    if not (abs(total_positive_weight - 1.0) < eps and abs(total_negative_weight - 1.0) < eps):
        msg = f"⚠️ Weight issue detected for {name}:\n"
        if abs(total_positive_weight - 1.0) >= eps:
            msg += f"  • Positive trait weights sum to {total_positive_weight:.4f} (should be 1.0000)\n"
        if abs(total_negative_weight - 1.0) >= eps:
            msg += f"  • Negative trait weights sum to {total_negative_weight:.4f} (should be 1.0000)\n"
        msg += "Please adjust the weights for this player."
        return None, msg
    
    score = (numerator / (1 + denominator)) * baseline
    return score, None


def compare_scores(players):
    results = []
    errors = []

    # Compute scores for all players
    for person in players:
        score, err = calculate_individual_score(
            person['name'],
            person['positive_traits'],
            person['negative_traits'],
            person['baseline']
        )
        if err:
            errors.append(err)
        else:
            results.append((person['name'], score * 100))  # store as percentage

    # Stop early if any errors found
    if errors:
        return "\n".join(errors)

    # Sort by descending score
    results.sort(key=lambda x: x[1], reverse=True)

    # Build leaderboard text
    leaderboard = "🏆 Should-I-Breakup Scoreboard 🏆\n"
    for rank, (name, score) in enumerate(results, start=1):
        leaderboard += f"{rank}. {name}: {score:.2f}%\n"

    # Winner highlight
    top_score = results[0][1]
    winners = [name for name, score in results if abs(score - top_score) < 1e-6]
    if len(winners) > 1:
        leaderboard += f"\n🤝 It's a tie between {', '.join(winners)}!"
    else:
        leaderboard += f"\n🎉 {winners[0]} wins!"

    return leaderboard


## Enter your players

In [46]:
player1 = {
    "name": "Sam",
    "positive_traits": {
        'Kind': (4, 0.25),
        'Masculine': (5, 0.25),
        'Family-centered': (5, 0.25),
        'Confident': (4, 0.25),
    },
    "negative_traits": {
        'Culture barriers': (2, 0.2),
        'Too many girl friends': (5, 0.3),
        'Bad memory': (2, 0.3),
        'Indecisive': (4, 0.2)
    },
    "baseline": 0.9 # Slightly happier because he hoops 🏀
}

player2 = {
    "name": "Owen",
    "positive_traits": {
        'Kind': (5, 0.3),
        'Religious': (5, 0.2),
        'Fun-loving': (4, 0.1),
        'Stable career': (3.5, 0.4)
    },
    "negative_traits": {
        'Impatient': (2, 0.2),
        'Communication': (4, 0.3),
        'Messy': (2, 0.2),
        'Not ambitious': (3.5, 0.3)
    },
    
    "baseline": 0.75  
}

player3 = {
    "name": "Nate",
    "positive_traits": {
        'Adventurous': (4, 0.2),
        'Supportive': (5, 0.3),
        'Intelligent': (4.5, 0.3),
        'Creative': (4, 0.2)
    },
    "negative_traits": {
        'Stubborn': (3, 0.3),
        'Jealous': (4, 0.4),
        'Workaholic': (2, 0.2),
        'Poor listener': (3, 0.1)
    },
    "baseline": 0.65 
}

player4 = {
    "name": "George",
    "positive_traits": {
        'Empathetic': (5, 0.25),
        'Ambitious': (4, 0.25),
        'Humorous': (4.5, 0.25),
        'Loyal': (5, 0.25)
    },
    "negative_traits": {
        'Pessimistic': (3, 0.3),
        'Overthinks': (4, 0.3),
        'Forgetful': (2, 0.2),
        'Impatient': (3, 0.2)
    },  
    "baseline": 0.4
}

# load players into a list
players = [player1, player2, player3, player4]

print(compare_scores(players))

🏆 Should-I-Breakup Scoreboard 🏆
1. Sam: 94.19%
2. Owen: 79.63%
3. Nate: 68.87%
4. George: 45.12%

🎉 Sam wins!


| Score Range       | Interpretation           |
|------------------|--------------------------|
| $< 50\%$          | 🚩 Red flag. It's about time to retreat         |
| $50\% - 90\%$     | 🌝 Decent- could tip either way, so keep an eye on it  |
| $\geq 90\%$       | 🌲 Excellent performance! Keep growing the relationship     |